# Step 1 Imports, Setup

In [ ]:
import pandas as pd
import numpy as np
import requests

In [ ]:
df = pd.read_csv("Cleaned_Saniya_23_25.csv") 
df.head()

In [ ]:
df.shape

## Dealing with Missing Street Addresses

In [ ]:
df['Application St Address'].isna().sum()

In [ ]:
df.loc[df['Application St Address'].isna()] # 26 na st addresses
# df.loc[df['Application St Address'].isna()]

In [ ]:
# Set Application Unit = 'Unit 109' for rows with IDs 524616 and 336451 --> these two rows are missing Unit numbers and they had only one application each 
df.loc[df['ID Number'].isin([524616, 336451]), 'Application Unit'] = 'Unit 109'

# normalize texts
df['Application Unit'] = df['Application Unit'].str.strip().str.lower()

# Drop the row with ID Number 134565 --> only had one application with no address, can be removed 
df = df[df['ID Number'] != 134565].reset_index(drop=True)


In [ ]:
# Fill in missing street addresses matched with resale data intfo 
df.loc[df['Application St Address'].isna() & (df['Application Unit'] == 'unit 109'), 'Application St Address'] = '1 Harvest Drive'



In [ ]:
df['Application St Address'].isna().sum()

# Step 2. Creating a new csv with addresses

In [ ]:
df = df.reset_index(drop=True)
df_address = pd.DataFrame({
    "Unique ID": range(1, len(df) + 1),
    "Street Address": df["Application St Address"],
    "City": df["Application City/Town"],
    # State filled as MA for now
    "State": ["MA"] * len(df),
    "ZIP": [None] * len(df)
})

df_address.to_csv("addresses.csv", index=False, header=False)

In [ ]:
df_address['Street Address'].isna().sum() # no more empty street addresses


# Step 3: Batch Addresses - standardizing addresses

In [ ]:
import csv
from io import StringIO

API_URL = "https://geocoding.geo.census.gov/geocoder/geographies/addressbatch"
params = {
    'benchmark': 'Public_AR_Current',
    'vintage': 'Current_Current',
}
with open('addresses.csv', 'rb') as file:
    files = {'addressFile': file}
    response = requests.post(API_URL, data=params, files=files)

    if response.status_code == 200:
        print("Geocoding successful. Generating results...")
        results = csv.reader(StringIO(response.text))
        response_rows = list(results)  # convert to list so we can iterate multiple times
        # Build dict keyed by UID (int)
        response_dict = {int(row[0]): row for row in response_rows}
        # Prepare lists for new columns
        match_statuses = []
        match_types = []
        matched_addresses = []

        # Assuming df_address has rows matching UID 1..N
        for uid in range(1, len(df_address) + 1):
            resp = response_dict.get(uid)
            if resp and len(resp) > 2:
                match_statuses.append(resp[2])  # Match Status
                match_types.append(resp[3] if len(resp) > 3 else None)  # Match Type
                matched_addresses.append(resp[4] if len(resp) > 4 else None)  # Matched Address
            else:
                match_statuses.append(None)
                match_types.append(None)
                matched_addresses.append(None)

        # Add these new columns to the dataframe
        df_address["Match Status"] = match_statuses
        df_address["Match Type"] = match_types
        df_address["Matched Address"] = matched_addresses
        df_address.to_csv("matched_addresses.csv", index=False)
    else:
        print("Request failed:", response.status_code)

In [ ]:
df_address.groupby(['Match Status']).describe()

In [ ]:
df_address.groupby(['Match Status', 'Match Type']).describe()

# Step 4: Accessing the Corresponding MSAs one row at a time

Now, for every row in our updated dataframe Send a request to the api using this matched address Handle appropriately Store the Metropolitan Statistical or Combined Statistical Area (new census category - msa is outdated?) in a column Leave null otherwise

In [ ]:
df_address.describe(include="all")

In [ ]:
def getMSA(matched_address):
    params = {
        'address': matched_address,
        'benchmark': 'Public_AR_Current',
        'vintage': 'Current_Current',
        'layers': '92,93',
        'format': 'json'
    }
    if (matched_address == None):
        print("No Matched Address from previous call")
        return
    response = requests.get('https://geocoding.geo.census.gov/geocoder/geographies/onelineaddress', params=params)
    if response.status_code == 200:
        data = response.json()
        matches = data.get('result', {}).get('addressMatches', [])
        if matches:
            geographies = matches[0].get('geographies', {})
            msa_info = geographies.get('Metropolitan Statistical Areas', [])
            if msa_info:
                return(msa_info[0]['BASENAME'])
            else:
                return None
        else:
            return None
    else:
        print("Request failed:", response.status_code)
        return None

In [ ]:
# Example:
i = 105
print(df_address.iloc[i]['Matched Address'])
test_address = df_address.iloc[i]['Matched Address'];
print(getMSA(test_address))

In [ ]:
#making a UID column for DF as well to later merge with the address data
df['Unique ID'] = range(1, len(df) + 1)

# Step 5: Creating a new csv from your dataframe


In [ ]:
from functools import lru_cache

# Optional: cache MSA lookups to avoid recomputing the same address
@lru_cache(maxsize=None)
def getMSA_cached(address):
    return getMSA(address)

# Apply function across valid rows
def resolve_msa(row):
    matched_address = row.get("Matched Address")
    if pd.notnull(matched_address) and isinstance(matched_address, str) and matched_address.strip():
        try:
            return getMSA_cached(matched_address)
        except:
            return None
    return None

# Apply the function vectorized-style
df_address["MSA"] = df_address.apply(resolve_msa, axis=1)


In [ ]:
# df_address["MSA"] = None  # Ensure the column exists

# for i, row in df_address.iterrows():
#     matched_address = row.get("Matched Address")
#     uid = row["Unique ID"]

#     if pd.notnull(matched_address) and isinstance(matched_address, str) and matched_address.strip():
#         msa_name = getMSA(matched_address)
#         df_address.loc[df_address["Unique ID"] == uid, "MSA"] = msa_name
#     else:
#         print(f"Skipping UID {uid} — no valid matched address.")

In [ ]:
df_address.to_csv("adresses_MSA.csv")

# Step 6 Merging the dataframe with your original csv

In [ ]:
df_merged = pd.merge(df, df_address, on="Unique ID", how="left")
df_merged.to_csv("data_updated.csv", index=False)

In [ ]:
df_merged.columns

## Dealing with missing MSA values

In [ ]:
df_merged['MSA'].isna().sum()  # Check how many rows have no MSA    # 168 rows without MSA 

In [ ]:
df_merged.loc[df_merged['MSA'].isna(), ['Unique ID', 'Application St Address', 'Application City/Town', 'Application Unit']]

In [ ]:
df_merged['Application St Address'] = df['Application St Address'].str.strip()

In [ ]:
df_merged.loc[df_merged['MSA'].isna(), 'Application St Address'].unique()  # Check unique addresses with no MSA


In [ ]:
df_merged.loc[df_merged['MSA'].isna(), 'Application City/Town'].unique()  # Check unique addresses with no MSA

In [ ]:
df_merged.loc[df_merged['MSA'].isna() & (df_merged['Application City/Town'] == 'North Andover'), 'Application St Address'].unique()  
# Fill in missing MSA for North Andover addresses
df_merged.loc[df_merged['MSA'].isna() & (df_merged['Application City/Town'] == 'North Andover'), 'MSA'] = 'Boston-Cambridge-Newton, MA–NH'


In [ ]:
df_merged.loc[df_merged['MSA'].isna() & (df_merged['Application City/Town'] == 'Edgartown'), 'Application St Address'].unique()  
df_merged.loc[df_merged['MSA'].isna() & (df_merged['Application City/Town'] == 'Edgartown'), 'MSA'] = 'N/A: Dukes County Vineyard Haven'

In [ ]:
df_merged.loc[df_merged['MSA'].isna() & (df_merged['Application City/Town'] == 'Falmouth'), 'Application St Address'].unique()  
df_merged.loc[df_merged['MSA'].isna() & (df_merged['Application City/Town'] == 'Falmouth'), 'MSA'] = 'Barnstable Town, MA'

In [ ]:
df_merged.loc[df_merged['MSA'].isna() & (df_merged['Application City/Town'] == 'Tyngsborough'), 'Application St Address'].unique()  
df_merged.loc[df_merged['MSA'].isna() & (df_merged['Application City/Town'] == 'Tyngsborough'), 'MSA'] = 'Boston-Cambridge-Newton, MA–NH'

In [ ]:
df_merged.loc[df_merged['MSA'].isna() & (df_merged['Application City/Town'] == 'Norfolk'), 'Application St Address'].unique()  
df_merged.loc[df_merged['MSA'].isna() & (df_merged['Application City/Town'] == 'Norfolk'), 'MSA'] = 'Boston-Cambridge-Newton, MA–NH'

In [ ]:
df_merged['MSA'].isna().sum()  # Check how many rows have MSA as NaN

In [ ]:
# save to a new CSV file
df_merged.to_csv("data_updated.csv", index=False)